In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("diabetes_dataset.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,2.0,3.0,4.0,2.0,888.0,888.0,3.0
1,1.0,3.0,4.0,88.0,888.0,888.0,3.0
2,2.0,3.0,2.0,2.0,101.0,101.0,3.0
3,2.0,4.0,4.0,88.0,888.0,230.0,3.0
4,1.0,4.0,4.0,25.0,301.0,888.0,3.0


In [3]:
df=df[df['DIABETE4'].isin([1,3])]

**Split Training and Test Set**

In [5]:
# Seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [7]:
print('Training set size: {}'.format(len(X_train)))
print('Test set size: {}'.format(len(X_test)))

Training set size: 80416
Test set size: 20104


**Building a Naive Bayes Model**

In [9]:
def predict_logprob(X, W, W_prior):
    '''
    Computes the log probability of each class given input features.
    Input:
        - X: input features
        - W: model weights
        - W_prior: weight prior probabilities
    Output:
        - y_pred: probability of positive product review
    '''
    y_pred=None
    # Ensure features are positive for NB, negative likelihood doesnt make sense. 
    X = X - X.min()
    # Compute log probability using dot product of features 
    #   with log of class conditional probabilities
    y_pred = np.dot(X, np.log(W.T))
    # Add log prior probabilities of classes
    y_pred += np.log(W_prior)
    return y_pred

In [10]:
def predict_probability(X, W, W_prior, classes):
    '''
    Produces probabilistic estimate for P(y_i = +1 | x_i)
        Estimate ranges between 0 and 1.
    Input:
        - X: Input features
        - W: model weights
        - W_prior: weight prior probabilities
        - classes: class labels for input features
    Output:
        - y_pred: probability of positive product review
    '''
    y_pred=None
    # Ensure features are positive for NB
    X = X - X.min()
        
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    # Identify index for positive class
    pos_class_idx = np.where(classes == 1)[0][0]
    # Get log probabilities
    y_pred = predict_logprob(X, W, W_prior)
    # Convert log probabilities to probabilities using the softmax function
    probs = np.exp(np.array(y_pred))
    probs = np.exp(probs) / np.sum(np.exp(probs),axis=1)[:, None]
    # Extract probability of positive class
    y_pred = probs[:, pos_class_idx]
    return y_pred

In [11]:
def predict(X, W, W_prior, classes):
    '''
    Hypothetical function  h(x)
    Input: 
        - X: Input features
        - W: model weights
        - W_prior: weight prior probabilities 
        - classes: class labels for input features
    Output:
        - y_pred: list of predicted classes 
    '''
    y_pred=None
    # Ensure features are positive for NB
    X = X - X.min()
    # Compute log probabilities
    y_pred = predict_logprob(X, W, W_prior)
    # Select class with highest probability
    y_pred = np.argmax(y_pred, axis=1)
    # Convert indices to class labels
    mapping = {i: k for i, k in enumerate(classes)}
    idx_to_class = np.vectorize(mapping.get)
    y_pred = idx_to_class(y_pred)
    return y_pred

In [12]:
def fit(X, Y, alpha=1):
    '''
    Compute closed-form solution for Naive Bayes classifier
    Input: 
        - X: Input features
        - Y: list of actual product sentiment classes 
        - alpha: laplace smoothing parameter
    Output:
        - W: predicted weights
        - W_prior: prior
        - likelihood_history: word likelihood (float)
    '''
    # Number of examples, Number of features
    num_examples, num_features = X.shape
    classes = np.unique(np.ravel(Y))
    num_classes = len(classes)
    # Initialization: weights, prior, likelihood
    W = np.zeros((num_classes, num_features))
    W_prior = np.zeros(num_classes)
    likelihood_history=[]

    # Compute class-conditional probabilities and class priors
    for ind, class_k in enumerate(classes):
        # Select samples belonging to class_k
        X_class_k = X[Y == class_k]
        # Compute likelihood using Laplace smoothing
        W[ind] = (np.sum(X_class_k, axis=0) + alpha)
        W[ind] /= (np.sum(X_class_k) + (alpha * X_class_k.shape[-1]))
        # Compute prior
        W_prior[ind] = X_class_k.shape[0] / num_examples
    
    # Compute and store log likelihood history
    log_likelihood = np.log(predict_probability(X, W, W_prior, classes)).mean()
    likelihood_history.append(log_likelihood)
    return W, W_prior, likelihood_history

In [13]:
model_weights, model_weights_prior, likelihood_history = fit(X_train, np.ravel(y_train))

/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [14]:
print("Model weights:", model_weights)

Model weights: [[0.99968392 0.99987253 0.99987099 0.99999261 0.99999937 0.99999933]
 [0.99993877 0.9999753  0.99997884 0.9999987  0.99999987 0.99999986]]


In [15]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.14480949 0.85519051]


In [16]:
print("Likelihood_history:", likelihood_history)

Likelihood_history: [-1.110116200789439]


**Classification Accuracy**

In [18]:
def get_classification_accuracy(prediction_labels, true_labels):    
    # Compute the number of correctly classified examples
    num_correct = np.sum(prediction_labels == true_labels)

    # Then compute accuracy by dividing num_correct by total number of examples
    accuracy = num_correct / len(true_labels)
    return accuracy

In [19]:
accuracy = get_classification_accuracy(predict(X_train, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_train))),
                                       np.ravel(y_train))
print(accuracy)

0.8551905093513729


In [20]:
accuracy = get_classification_accuracy(predict(X_test, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_test))),
                                       np.ravel(y_test))
print(accuracy)

0.857391563867887


**Classification Precision**

In [22]:
y_pred = predict(X_test, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [23]:
np.unique(y_pred)

array([3.])

In [24]:
print(len(y_pred), len(y_test))

20104 20104


In [25]:
y_pred = np.array(y_pred).flatten()

true_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 1:
        true_pos += 1

print(true_pos)

0


In [26]:
false_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 3 and y_pred[i] == 1:
        false_pos += 1

print(false_pos)

0


In [27]:
precision = true_pos/(true_pos+false_pos)
print("Precision on test data: %s" % precision)

ZeroDivisionError: division by zero

**Recall**

In [49]:
false_neg = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 3:
        false_neg += 1

print(false_neg)

2867


In [50]:
recall = true_pos / (true_pos + false_neg)
print("Recall on test data: %s" % recall)

Recall on test data: 0.0


In [51]:
print(np.unique(y_train))
print(np.unique(y_test))
print(np.unique(y_pred))

[1. 3.]
[1. 3.]
[3.]


In [52]:
from collections import Counter
print("y_train distribution:", Counter(y_train))
print("y_test distribution:", Counter(y_test))

y_train distribution: Counter({'DIABETE4': 1})
y_test distribution: Counter({3.0: 17237, 1.0: 2867})


In [53]:
print(model_weights_prior) #Class 3 prior much larger than class 1

[0.14480949 0.85519051]


**Undersample Majority Class**

In [55]:
# Convert to numpy
X_train = np.array(X_train)
y_train = np.array(y_train).flatten()

# Separate classes
X_majority = X_train[y_train == 3]
X_minority = X_train[y_train == 1]

y_majority = y_train[y_train == 3]
y_minority = y_train[y_train == 1]

# Get minority size
n_minority = len(X_minority)

# Randomly sample from majority WITHOUT replacement
indices = np.random.choice(len(X_majority), size=n_minority, replace=False)

X_majority_downsampled = X_majority[indices]
y_majority_downsampled = y_majority[indices]

# Combine
X_train_balanced = np.vstack((X_majority_downsampled, X_minority))
y_train_balanced = np.hstack((y_majority_downsampled, y_minority))

# Shuffle
shuffle_idx = np.random.permutation(len(y_train_balanced))
X_train_balanced = X_train_balanced[shuffle_idx]
y_train_balanced = y_train_balanced[shuffle_idx]

In [56]:
from collections import Counter
print(Counter(y_train_balanced))

Counter({3.0: 11645, 1.0: 11645})


In [57]:
model_weights, model_weights_prior, likelihood_history = fit(X_train_balanced, np.ravel(y_train_balanced))

In [58]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.5 0.5]


In [59]:
accuracy = get_classification_accuracy(predict(X_train_balanced, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_train_balanced))),
                                       np.ravel(y_train_balanced))
print(accuracy)

0.5738514383855732


In [60]:
accuracy = get_classification_accuracy(predict(X_test, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_test))),
                                       np.ravel(y_test))
print(accuracy)

0.5271090330282531


In [61]:
y_pred = predict(X_test, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [62]:
np.unique(y_pred)

array([1., 3.])

In [63]:
y_pred = np.array(y_pred).flatten()

true_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 1:
        true_pos += 1

print(true_pos)

1935


In [64]:
false_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 3 and y_pred[i] == 1:
        false_pos += 1

print(false_pos)

8575


In [65]:
precision = true_pos/(true_pos+false_pos)
print("Precision on test data: %s" % precision)

Precision on test data: 0.1841103710751665


In [66]:
false_neg = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 3:
        false_neg += 1

print(false_neg)

932


In [67]:
recall = true_pos / (true_pos + false_neg)
print("Recall on test data: %s" % recall)

Recall on test data: 0.6749215207534007


In [68]:
if precision + recall == 0:
    f1 = 0.0
else:
    f1 = 2 * (precision * recall) / (precision + recall)

print("F1 score:", f1)

F1 score: 0.2893025342004934


**Pre-Processing Oversampling**

In [70]:
df=pd.read_csv("data_oversampling.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,1.0,4.0,4.0,88.0,888.0,888.0,3.0
1,1.0,4.0,4.0,88.0,888.0,888.0,1.0
2,1.0,2.0,4.0,2.0,301.0,107.0,3.0
3,1.0,4.0,4.0,88.0,101.0,101.0,1.0
4,1.0,2.0,4.0,88.0,301.0,888.0,3.0


In [71]:
df=df[df['DIABETE4'].isin([1,3])]

In [72]:
# Seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [73]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [74]:
print('Training set size: {}'.format(len(X_train)))
print('Test set size: {}'.format(len(X_test)))

Training set size: 110033
Test set size: 27509


In [75]:
np.unique(y_pred)

array([1., 3.])

In [76]:
#normalize using training stats
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [77]:
X_train_scaled.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4
61518,-0.612973,-0.189377,0.700619,-1.314758,-0.947178,0.890556
129727,-0.612973,1.028705,-0.434996,-1.314758,0.776210,0.890556
96362,-0.612973,-0.189377,0.700619,0.814066,0.776210,-1.322642
125189,-0.612973,-0.189377,0.700619,0.814066,-1.001792,-1.024170
14258,-0.612973,1.028705,0.700619,0.814066,0.776210,0.890556


In [78]:
model_weights, model_weights_prior, likelihood_history = fit(X_train_scaled, np.ravel(y_train))

/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [79]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.50017722 0.49982278]


In [80]:
X_train_pred = predict(X_train_scaled, model_weights, model_weights_prior, np.unique(np.ravel(y_train))), np.ravel(y_train)

In [81]:
y_pred = predict(X_test_scaled, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [82]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 3) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 3))
    tn = np.sum((y_true == 3) & (y_pred == 3))

    if (tp + fp) != 0:
        precision = tp / (tp + fp)
    else:
        precision = 0

    if (tp + fn) != 0:
        recall = tp / (tp + fn)  
    else: 
        recall = 0
    
    if (precision + recall) != 0:
        f1_score = (2 * precision * recall) / (precision + recall)
    else: 
        f1_score = 0
    
    if (tp + fp + fn + tn) != 0:
        accuracy = (tp + tn) / (tp + fp + fn + tn)  
    else:
        accuracy = 0

    return accuracy, precision, recall, f1_score, tp, fp, fn, tn

In [83]:
accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test, y_pred)

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1_score)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

accuracy: 0.45257915591261044
precision: 0.40936473165388826
recall: 0.21769202766654533
f1 score: 0.28423404154189835
TP: 2990
FP: 4314
FN: 10745
TN: 9460


**Pre-processing Under-Sampling**

In [85]:
df=pd.read_csv("data_undersampling.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,1.0,4.0,3.0,30.0,888.0,888.0,3.0
1,1.0,4.0,4.0,20.0,888.0,888.0,3.0
2,1.0,4.0,3.0,88.0,888.0,203.0,1.0
3,2.0,4.0,3.0,88.0,203.0,888.0,1.0
4,1.0,2.0,3.0,88.0,103.0,888.0,1.0


In [86]:
df=df[df['DIABETE4'].isin([1,3])]

In [87]:
# Seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [88]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [89]:
print('Training set size: {}'.format(len(X_train)))
print('Test set size: {}'.format(len(X_test)))

Training set size: 18632
Test set size: 4658


In [90]:
#normalize using training stats
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [91]:
X_train_scaled.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4
16921,-0.615633,-0.195434,-1.582570,-1.109365,-1.307139,-1.040461
21098,-0.615633,1.028130,-2.721218,0.810107,0.775857,0.889580
16138,-0.615633,1.028130,0.694727,0.810107,0.775857,0.889580
13234,-0.615633,-1.418998,0.694727,0.810107,0.775857,-0.964386
14460,-0.615633,-1.418998,-0.443921,-1.240836,0.775857,0.889580


In [92]:
model_weights, model_weights_prior, likelihood_history = fit(X_train_scaled, np.ravel(y_train))

/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [93]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.49774581 0.50225419]


In [94]:
y_pred = predict(X_test_scaled, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [95]:
accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test, y_pred)

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1_score)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

accuracy: 0.4394589952769429
precision: 0.3250728862973761
recall: 0.09405314213412062
f1 score: 0.14589466797513903
TP: 223
FP: 463
FN: 2148
TN: 1824


**SMOTE**

In [146]:
df=pd.read_csv("data_smote.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,1.0,3.0,4.0,88.0,888.0,105.0,3.0
1,1.0,3.0,4.0,88.0,888.0,204.0,3.0
2,1.0,3.0,3.0,88.0,888.0,201.0,3.0
3,1.0,3.0,4.0,88.0,301.0,888.0,3.0
4,2.0,4.0,1.0,88.0,888.0,203.0,1.0


In [148]:
df=df[df['DIABETE4'].isin([1,3])]

In [150]:
# Seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [152]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [154]:
print('Training set size: {}'.format(len(X_train)))
print('Test set size: {}'.format(len(X_test)))

Training set size: 110033
Test set size: 27509


In [156]:
#normalize using training stats
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [158]:
X_train_scaled.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4
61518,1.667793,-1.432780,-0.451157,-0.706349,-0.987039,-0.987068
129727,1.169639,1.035410,0.451588,-0.706349,0.774539,-1.015215
96362,-0.619454,0.603581,0.702948,0.816600,-1.305397,-1.037732
125189,0.766224,-0.198685,-0.451157,0.816600,0.774539,-1.029288
14258,-0.619454,1.035410,-0.451157,0.816600,0.774539,0.893156


In [160]:
model_weights, model_weights_prior, likelihood_history = fit(X_train_scaled, np.ravel(y_train))

/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [162]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.50058619 0.49941381]


In [164]:
y_pred = predict(X_test_scaled, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [166]:
accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test, y_pred)

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1_score)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

accuracy: 0.47537169653567923
precision: 0.45867676542659835
recall: 0.3008035062089116
f1 score: 0.3633315687312511
TP: 4118
FP: 4860
FN: 9572
TN: 8959


**Prep for Streamlit**

In [171]:
best_nb_model_final = {
    "model_name": "Best NB Model Final",
    "dataset_method": "smote",
    "prior": model_weights_prior,
    "mean": mean,
    "std": std,
}

best_nb_model_final

{'model_name': 'Best NB Model Final',
 'dataset_method': 'smote',
 'prior': array([0.50058619, 0.49941381]),
 'mean': EXERANY2      1.270830
 _BMI5CAT      3.160997
 _SMOKER3      3.390915
 MENTHLTH     56.900594
 SSBFRUT3    632.543210
 ALCDAY4     570.682238
 dtype: float64,
 'std': EXERANY2      0.437207
 _BMI5CAT      0.810310
 _SMOKER3      0.866472
 MENTHLTH     38.084016
 SSBFRUT3    329.817942
 ALCDAY4     355.276847
 dtype: float64}

In [173]:
nb_explore_models = {}

#What is datasets.items()?
for dataset_name, (X_train_dataset, y_train_dataset) in datasets.items():
    nb_explore_models[dataset_name] = {}

    trained_model = predict(X_train_dataset, model_weights, model_weights_prior, np.unique(np.ravel(y_train_dataset)))
    trained_model.fit(X_train_dataset, y_train_dataset)

    y_pred = trained_model.predict(X_test_final)
    _accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test_final_svm, y_pred)

    svm_explore_models[dataset_name] = {
        "model_name": f"NB ({dataset_name})",
        "dataset_method": dataset_name,
        "prior": model_weights_prior,
        "mean": mean,
        "std": std,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

NameError: name 'datasets' is not defined